> 在线讲义：[第 53 章 · 专家放置、复制与负载均衡：最忙的那张卡定完成时刻](https://wikieden.github.io/ai-infra-from-constraints/lessons/53-expert-balance.html) · 本书《从约束推导 AI Infra》

# 第 53 章 · 专家放置、复制与负载均衡：最忙的那张卡定完成时刻

配套讲义：仓库里 `ai-infra/notes/lessons/53-expert-balance.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



大 EP 的 skew、例 9.5（两台 HGX H100 的 dispatch / combine）、例 9.6（专家副本回本）与 Grouped GEMM 补齐，取自 AII 第 9 章 §9.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import random
MiB = 2**20
H100 = 494.7e12                       # BF16 峰值的 50%
FLOP_PER = 6 * 4096 * 1536            # 一份 token—专家任务
ROW = 8 * 1024                        # 每行 8 KiB

**函数 `hot_skew`**：一个热点专家（hot 行）、其余均分：返回 (热点卡行数, 热点卡 / 平均)。

In [ ]:
def hot_skew(ep: int, n: int = 1024, k: int = 8, E: int = 256, hot: int = 128) -> tuple[float, float]:
    """一个热点专家（hot 行）、其余均分：返回 (热点卡行数, 热点卡 / 平均)。"""
    rest = (n * k - hot) / (E - 1)
    per_card = E // ep
    rows = hot + (per_card - 1) * rest
    return rows, rows / (n * k / ep)

**函数 `random_skew`**：均匀随机路由下 最忙卡 / 平均 的批平均。

In [ ]:
def random_skew(ep: int, batches: int = 1000, n: int = 1024, k: int = 8, E: int = 256, seed: int = 0) -> float:
    """均匀随机路由下 最忙卡 / 平均 的批平均。"""
    rng = random.Random(seed)
    per_card = E // ep
    tot = 0.0
    for _ in range(batches):
        cards = [0] * ep
        for _ in range(n):
            for e in rng.sample(range(E), k):
                cards[e // per_card] += 1
        tot += max(cards) / (n * k / ep)
    return tot / batches

**函数 `exchange`**：例 9.5：(dispatch, 最忙卡计算, combine, 屏障总和) ms。shared 是两池间总带宽上限。

In [ ]:
def exchange(groups: list[int], bw: float = 50e9, dispatch_bytes: int = ROW, combine_bytes: int = ROW, shared: float | None = None):
    """例 9.5：(dispatch, 最忙卡计算, combine, 屏障总和) ms。shared 是两池间总带宽上限。"""
    total = sum(groups)
    d = max(max(groups) * dispatch_bytes / bw, total * dispatch_bytes / shared if shared else 0)
    c = max(max(groups) * combine_bytes / bw, total * combine_bytes / shared if shared else 0)
    comp = max(groups) * FLOP_PER / H100
    return tuple(round(x * 1e3, 3) for x in (d, comp, c, d + comp + c))

**函数 `replica_breakeven`**：(复制时间, 回本批数)：N_min = ⌊T_copy / Δt⌋ + 1。

In [ ]:
def replica_breakeven(dt: float, n_copies: int = 7, bw: float = 450e9, alpha: float = 5e-6, w: float = 36 * MiB):
    """(复制时间, 回本批数)：N_min = ⌊T_copy / Δt⌋ + 1。"""
    t = n_copies * (w / bw + alpha)
    return t, math.floor(t / dt) + 1

**函数 `padded_rows`**：(非空专家各补到 tile 的倍数, 全部补到最大值)。

In [ ]:
def padded_rows(rows: list[int], tile: int = 32) -> tuple[int, int]:
    """(非空专家各补到 tile 的倍数, 全部补到最大值)。"""
    a = sum(math.ceil(r / tile) * tile for r in rows if r)
    return a, len(rows) * max(rows)

**函数 `three_stage`**：三段流水：一个 micro-batch 三段之和 + (q − 1) × 最长段；slow 是通信被拖慢的倍数。

In [ ]:
def three_stage(d: float, c: float, m: float, q: int = 2, slow: float = 1.0) -> float:
    """三段流水：一个 micro-batch 三段之和 + (q − 1) × 最长段；slow 是通信被拖慢的倍数。"""
    s = [d / q * slow, c / q, m / q * slow]
    return sum(s) + (q - 1) * max(s)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：总量一样，最忙的卡定完成时刻")
gf = 512 * 2 * 18874368
print(f"  512 次分派 {gf / 1e9:.1f} GFLOPs：均分每卡 {gf / 8 / H100 * 1e6:.1f} μs，全在一张卡 {gf / H100 * 1e6:.1f} μs")
for ep in (8, 32, 256):
    r, s = hot_skew(ep)
    print(f"  EP{ep}：热点卡 {r:.0f} 行，是平均的 {s:.2f} 倍")
print(f"  均匀随机（1000 批）：EP8 {random_skew(8):.2f}，EP256 {random_skew(256):.2f}")

In [ ]:
print("\n第二步：例 9.5（ms：dispatch / 计算 / combine / 合计）")
bal, hot = [2048] * 4, [5120, 1024, 1024, 1024]
print(f"  均衡跨服务器     {exchange(bal)}")
print(f"  热点跨服务器     {exchange(hot)}")
print(f"  均衡、只剩一条链路 {exchange(bal, shared=50e9)}")
print(f"  热点、dispatch FP8 {exchange(hot, dispatch_bytes=ROW // 2)}")
print(f"  热点、同一台 NVLink {exchange(hot, bw=450e9)}")

In [ ]:
print("\n第三步：例 9.6 专家副本")
before = 921e6 / 1675e9
dt = before - before * 576 / 1024
for name, bw in (("NVLink 450 GB/s", 450e9), ("网卡 50 GB/s", 50e9)):
    t, n = replica_breakeven(dt, bw=bw)
    print(f"  {name}：复制 {t * 1e3:.3f} ms，每批省 {dt * 1e3:.3f} ms，第 {n} 批回本；16 批净 {(16 * dt - t) * 1e3:+.1f} ms，64 批 {(64 * dt - t) * 1e3:+.1f} ms")
print(f"  每卡每层一个副本：94 × 36 MiB = {94 * 36 / 1024:.2f} GiB；只给 8 层：{8 * 36} MiB")

In [ ]:
print("\n第四步：补齐与重叠")
print(f"  [32,16,8,4,2,1,1,0] 共 64 行：补到 32 的倍数 {padded_rows([32, 16, 8, 4, 2, 1, 1, 0])[0]}，全补到最大 {padded_rows([32, 16, 8, 4, 2, 1, 1, 0])[1]}")
print(f"  0.336 / 0.156 / 0.336：串行 0.827，两个 micro-batch {three_stage(0.336, 0.156, 0.336):.3f}，通信慢 49% {three_stage(0.336, 0.156, 0.336, slow=1.49):.3f} ms")

## 3. 代码里才看得到的

1. `random_skew` 用固定种子 0 跑 1000 批，EP256 得 1.53（原文 1.52）；换种子会差一两个百分点。
2. `exchange` 的 `shared` 参数把总量除以两池间带宽，再和单卡收发取最大，这就是割集那一项。
3. `three_stage` 算出 0.5813 ms，四舍五入成 0.581（原文）或 0.582（三位小数的 round），差在第四位。

## 4. 自己改一改

- 给 `hot_skew` 加"热点专家有 r 个副本、行数均分"，看 EP256 要几个副本才压到 1.5 倍以下。
- 把 `exchange` 的 `bw` 改成 25e9，看哪一行最先超过 3 ms。
- 用 `three_stage` 扫 q = 1…8，看 micro-batch 拆多细收益就停了。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 小组作业看最后一个交的人；
> 能吃的人一人一桌，就是别人的四倍；
> 包裹都寄一家，送和收两趟都堵他门口；
> 多开窗口先搬桌子，高峰够长才回本；
> 一个人也开一整车，等最慢的那条分支。